<a href="https://colab.research.google.com/github/jarekwan/praca_inzynierska/blob/main/predict_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/ml_project', exist_ok=True)
print("folder ready")

import sys
sys.path.append('/content/drive/MyDrive/ml_project')

Mounted at /content/drive
folder ready


In [ ]:
%%writefile /content/drive/MyDrive/ml_project/predict_ml.py
# -*- coding: utf-8 -*-
import os
import pickle
import pandas as pd
import numpy as np

target_dir = "/content/drive/MyDrive/ml_project"

model_path = os.path.join(target_dir, "ml_trained.pkl")
x_test_path = os.path.join(target_dir, "X_test.pkl")
y_pred_path = os.path.join(target_dir, "y_pred_ml.pkl")

# ---------------------------------------------------------
# predict_ml
# ---------------------------------------------------------
def predict_ml():

    # check files
    if not os.path.exists(model_path):
        raise FileNotFoundError("ml_trained.pkl not found")

    if not os.path.exists(x_test_path):
        raise FileNotFoundError("X_test.pkl not found")

    # load trained model
    with open(model_path, "rb") as f:
        model = pickle.load(f)

    # load test features
    X_test = pd.read_pickle(x_test_path)

    # predict
    y_pred = model.predict(X_test)

    # convert to numpy array
    y_pred = np.array(y_pred)

    # save predictions
    with open(y_pred_path, "wb") as f:
        pickle.dump(y_pred, f)

    print("saved:", y_pred_path)
    print("ml predictions generated")

    return y_pred


Overwriting /content/drive/MyDrive/ml_project/predict_ml.py


KROCZACY MODEL

In [ ]:
%%writefile /content/drive/MyDrive/ml_project/predict_ml.py
# -*- coding: utf-8 -*-
import os
import pickle
import pandas as pd
import numpy as np

target_dir = "/content/drive/MyDrive/ml_project"

model_path = os.path.join(target_dir, "ml_trained.pkl")
x_train_path = os.path.join(target_dir, "X_train.pkl")
x_test_path = os.path.join(target_dir, "X_test.pkl")
y_train_path = os.path.join(target_dir, "y_train.pkl")
y_test_path = os.path.join(target_dir, "y_test.pkl")
y_pred_path = os.path.join(target_dir, "y_pred_ml.pkl")

# ---------------------------------------------------------
# predict_ml  (ROLLING)
# ---------------------------------------------------------
def predict_ml():

    # check files
    for p in [model_path, x_train_path, x_test_path, y_train_path, y_test_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"{p} not found")

    # load model
    with open(model_path, "rb") as f:
        model = pickle.load(f)

    # load data
    X_train = pd.read_pickle(x_train_path).fillna(0)
    y_train = pd.read_pickle(y_train_path)

    X_test = pd.read_pickle(x_test_path).fillna(0)
    y_test = pd.read_pickle(y_test_path)

    y_pred = []

    # -----------------------------------------------------
    # rolling prediction
    # -----------------------------------------------------
    for t in range(len(X_test)):

        # 1. predict using x_test[t]
        x_t = X_test.iloc[[t]]
        y_hat = model.predict(x_t)[0]
        y_pred.append(y_hat)

        # 2. get real value AFTER prediction
        y_real = y_test.iloc[t]

        # 3. update training set
        X_train = pd.concat([X_train, x_t], axis=0)
        y_train = pd.concat([y_train, pd.Series([y_real])], axis=0)

        # 4. retrain model
        model.fit(X_train, y_train)

    y_pred = np.array(y_pred)

    # save predictions
    with open(y_pred_path, "wb") as f:
        pickle.dump(y_pred, f)

    print("saved:", y_pred_path)
    print("ml rolling predictions generated")

    return y_pred
